# 08. Confie no experimento

Antes de acreditar num resultado, vale checar se o experimento foi bem conduzido. A biblioteca traz três diagnósticos: **SRM** (a alocação saiu como o esperado?), **A/A** (o pipeline está calibrado?) e **balanço** (covariáveis equilibradas? visto no notebook 04).

## SRM: a alocação bate com o esperado?

Um experimento recém-randomizado bate por construção. Mas se o pipeline de dados perde unidades de um braço (logging, filtro de bots), a proporção observada destoa. O `SRMTest` pega isso.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.diagnostics import SRMTest

rng = np.random.default_rng(0)
n = 200
df = pd.DataFrame({"x": rng.normal(size=n)})
design = CRD(p=0.5, seed=0)
assignment = design.randomize(df)
print("Recem-randomizado, flag:", SRMTest().run(assignment).flagged)

# Simula perda de 40% dos controles (vazamento assimetrico de dados)
data = assignment.data_
treated = data[data["treatment"] == 1]
control = data[data["treatment"] == 0].sample(frac=0.6, random_state=0)
corrupted_df = pd.concat([treated, control]).reset_index(drop=True)
corrupted = CRDAssignment(data=corrupted_df, treatment_col="treatment", design=design, seed=0)
res = SRMTest().run(corrupted)
print(f"Apos perda de controles, flag: {res.flagged}  (p={res.p_value:.2e})")

## A/A: o pipeline está calibrado?

Um teste A/A re-randomiza sobre dados **sem efeito** e checa se a taxa de falso-positivo bate com `alpha`. Se destoar, há algo errado no desenho ou na inferência.

In [ ]:
from skxperiments.diagnostics import AATest
from skxperiments.estimators.difference_in_means import DifferenceInMeans
from skxperiments.inference import NeymanCI

aa_df = pd.DataFrame({"y": rng.normal(size=200)})   # outcome sem efeito
aa = AATest(
    design=CRD(p=0.5, seed=0),
    inference=NeymanCI(estimator=DifferenceInMeans(outcome_col="y")),
    n_simulations=300,
    seed=0,
)
res_aa = aa.run(aa_df)
print(
    f"Taxa de falso-positivo: {res_aa.false_positive_rate:.3f} "
    f"(alpha={res_aa.alpha}), flag={res_aa.flagged}"
)

## O que aprendemos

- **SRM** é um alarme barato e poderoso para bugs de coleta; rode sempre antes de analisar.
- **A/A** valida que design + inferência estão calibrados (falso-positivo perto de `alpha`).
- **Balanço** (notebook 04) completa o trio. No próximo notebook, o `ExperimentPipeline` roda o SRM automaticamente.

**Próximo:** `09. Juntando tudo` planeja, roda e relata um experimento de ponta a ponta.